# ETL MV Flow 恢复运行：复用前置 artifact 到新 run_id

这个 notebook 用于处理主流程在 batch workflow cell 中途失败后的恢复测试。它不会复用失败 cell 内产生的 `04_batch_mvs`、`05_rewritten_sql`、`06_execution_logs`，只复制该 cell 之前已经正确生成的前置 artifact：`00_raw_sql`、`01_query_blocks`、`03_batches`，并可选复制 `02_families` 作为旧 QueryFamily 参考。

当前测试链路按方案 A 运行：`QueryFamily` / `family_groups` 不再作为 BatchMVAgent 的硬边界。notebook 会清空复制过来的 legacy `family_groups`，并在执行 `BatchWorkflowRunner.run_all_batches(...)` 时不传 `families_path`，方便直接测试 batch-local QueryBlock MV 生成和 rewrite。


In [ ]:
from pathlib import Path
import shutil
import sys
import os
from datetime import datetime
from dotenv import load_dotenv

# 读取项目根目录 .env，并把项目根目录加入 Python import 路径。
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / "llm_demo").is_dir() and (path / "tpcds-spark").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

# 当前 workflow 需要多轮 LLM 调用；给 notebook 测试链路设置更宽松的网络超时默认值。
os.environ["LLM_CONNECT_TIMEOUT_SECONDS"] = "60"
os.environ["LLM_TIMEOUT_SECONDS"] = "180"
os.environ["LLM_MAX_RETRIES"] = "4"
PROJECT_ROOT


In [ ]:
import importlib

import llm_demo.src.agents.batch_mv_agent as batch_mv_agent_module
import llm_demo.src.agents.executor_agent as executor_agent_module
import llm_demo.src.agents.rewrite_agent as rewrite_agent_module
import llm_demo.src.agents.self_iteration_agent as self_iteration_agent_module
import llm_demo.src.core.batch_workflow_runner as batch_workflow_runner_module
import llm_demo.src.core.coverage_summary_builder as coverage_summary_builder_module

# Notebook kernel 会缓存已 import 的类；修改 agent 代码后重跑本 cell，确保使用磁盘上的最新实现。
for module in [
    batch_mv_agent_module,
    executor_agent_module,
    rewrite_agent_module,
    batch_workflow_runner_module,
    coverage_summary_builder_module,
    self_iteration_agent_module,
]:
    importlib.reload(module)

from llm_demo.src.agents.self_iteration_agent import SelfIterationAgent
from llm_demo.src.core.artifact_store import ArtifactStore
from llm_demo.src.core.llm_client import LLMClient
from llm_demo.src.core.batch_workflow_runner import BatchWorkflowRunner
from llm_demo.src.core.coverage_summary_builder import CoverageSummaryBuilder


In [ ]:
# 默认复用最近一次在 batch workflow cell 失败前已经生成的前置 artifact。
# 如果要复用其他 run，改这里即可。
SOURCE_RUN_ID = "20260612_151048"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_root = PROJECT_ROOT / "llm_demo" / "artifacts"
source_run_dir = artifact_root / SOURCE_RUN_ID
store = ArtifactStore(run_id=run_id, artifact_root=artifact_root)
llm_client = LLMClient(project_root=PROJECT_ROOT)

if not source_run_dir.is_dir():
    raise FileNotFoundError(f"source run not found: {source_run_dir}")

{
    "source_run_id": SOURCE_RUN_ID,
    "new_run_id": run_id,
    "new_run_dir": str(store.run_dir),
}


In [ ]:
# 只复制报错 cell 之前的正确模块产物。
# 不复制 04_batch_mvs / 05_rewritten_sql / 06_execution_logs，避免把半完成 batch 状态带入新 run。
# 02_families 只作为可选参考复制；当前 workflow 调用不会把它作为 BatchMV 硬边界传入。
REQUIRED_PREFLIGHT_ARTIFACT_DIRS = [
    "00_raw_sql",
    "01_query_blocks",
    "03_batches",
]
OPTIONAL_PREFLIGHT_ARTIFACT_DIRS = [
    "02_families",
]

store.run_dir.mkdir(parents=True, exist_ok=True)
copied_dirs = []
for relative_dir in REQUIRED_PREFLIGHT_ARTIFACT_DIRS:
    source_dir = source_run_dir / relative_dir
    target_dir = store.run_dir / relative_dir
    if not source_dir.is_dir():
        raise FileNotFoundError(f"required preflight artifact dir not found: {source_dir}")
    if target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.copytree(source_dir, target_dir)
    copied_dirs.append(relative_dir)

for relative_dir in OPTIONAL_PREFLIGHT_ARTIFACT_DIRS:
    source_dir = source_run_dir / relative_dir
    target_dir = store.run_dir / relative_dir
    if not source_dir.is_dir():
        continue
    if target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.copytree(source_dir, target_dir)
    copied_dirs.append(relative_dir)

{
    "copied_dirs": copied_dirs,
    "source_run_dir": str(source_run_dir),
    "new_run_dir": str(store.run_dir),
}


In [ ]:
# 复用后的核心路径。
# 当前方案 A 不要求 families_path；如果 02_families 被复制，仅保留为人工参考。
sql_manifest_path = store.sql_manifest_path
query_blocks_path = store.query_blocks_path
families_path = store.query_families_path if store.query_families_path.exists() else None
batches_path = store.complexity_batches_path

for path in [sql_manifest_path, query_blocks_path, batches_path]:
    if not path.exists():
        raise FileNotFoundError(path)

# 清空旧 artifact 中的 legacy family_groups，确保后续 MV 生成走 batch-local QueryBlock 边界。
batches_artifact = store.read_json(batches_path)
legacy_family_group_counts = {
    batch["batch_id"]: len(batch.get("family_groups", []))
    for batch in batches_artifact["complexity_batches"]
}
for batch in batches_artifact["complexity_batches"]:
    batch["family_groups"] = []
batches_path = store.write_json("03_batches/complexity_batches.json", batches_artifact)

{
    "sql_manifest_path": str(sql_manifest_path),
    "query_blocks_path": str(query_blocks_path),
    "families_path_optional_reference": str(families_path) if families_path else None,
    "batches_path": str(batches_path),
    "cleared_legacy_family_group_counts": legacy_family_group_counts,
}


In [ ]:
# 先确认 CSV batch artifact 的覆盖情况，再决定是否继续跑 batch workflow。
# family_group_count 预期为 0；BatchMVAgent 会使用 batch-local QueryBlock candidate groups。
batches = store.read_json(batches_path)["complexity_batches"]
[
    {
        "batch_id": batch["batch_id"],
        "batch_type": batch["batch_type"],
        "query_count": len(batch.get("query_ids", [])),
        "family_group_count": len(batch.get("family_groups", [])),
    }
    for batch in batches
]


In [ ]:
# 从空 materialized_mvs 状态开始，逐个非空 batch 重新执行完整 dry-run 闭环。
# 这里不传 families_path，用于验证 QueryFamily 不再是 BatchMVAgent 的硬边界。
workflow_outputs = BatchWorkflowRunner(store, llm_client).run_all_batches(
    complexity_batches_path=batches_path,
    sql_manifest_path=sql_manifest_path,
    query_blocks_path=query_blocks_path,
)
workflow_outputs[-5:]


In [ ]:
# 生成覆盖率摘要。
coverage_path = CoverageSummaryBuilder(store).run()
store.read_json(coverage_path)


In [ ]:
# 可选：生成 rules 反馈建议；不会自动修改 rules 文件。
feedback_path = SelfIterationAgent(store, llm_client).run(store.run_log_path)
store.read_json(feedback_path)
